# Install Libraries

* Pandas for manipulating data
* Numpy for numerical array
* Matplotlib and Seaborn for graphical represtations and EDA Purposes
   
 


In [ ]:
!pip install pandas numpy matplotlib seaborn

  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.0 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.0 MB 1.7 MB/s eta 0:00:06
   -- ------------------------------------- 0.5/10.0 MB 1.7 MB/s eta 0:00:06
   --- ------------------------------------ 0.8/10.0 MB 768.7 kB/s eta 0:00:12
   --- ------------------------------------ 0.8/10.0 MB 768.7 kB/s eta 0:00:12
   --- ------------------------------------ 0.8/10.0 MB 768.7 kB/s eta 0:00:12
   ---- ----------------------------------- 1.0/10.0 MB 646.0 kB/s eta 0:00:14
   ---- ----------------------------------- 1.0/10.0 MB 646.0 kB/s eta 0:00:14
   ----- ---------------------------------- 1.3/10.0 MB 604.2 kB/s eta 0:00:15
   ----- ---------------------------------- 1.3/10.0 MB 604.2 kB/s eta 0:00:15
   ----- ---------------------------------- 1.3/10.0 MB 604.2 kB/s eta 0:00:15
   ---

# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np

## Importing dataset and checking columns and rows count

In [9]:
file_path = "flights.csv"

flights = pd.read_csv(r"C:\Users\Shakthi\Desktop\Flitzz\data\flights.csv")

print("Data loaded successfully!")
print("Rows:", flights.shape[0])
print("Columns:", flights.shape[1])

C:\Users\Shakthi\AppData\Local\Temp\ipykernel_9316\2362150268.py:3: DtypeWarning: Columns (0: ORIGIN_AIRPORT, 1: DESTINATION_AIRPORT) have mixed types. Specify dtype option on import or set low_memory=False.
  flights = pd.read_csv(r"C:\Users\Shakthi\Desktop\Flitzz\data\flights.csv")


Data loaded successfully!
Rows: 5819079
Columns: 31


## Conversion of origin_airport and destination_airport column for obj to string

Reason: Warning of mixed data types but not a necessary step for data preparation

In [7]:
flights["ORIGIN_AIRPORT"] = flights["ORIGIN_AIRPORT"].astype("string")
flights["DESTINATION_AIRPORT"] = flights["DESTINATION_AIRPORT"].astype("string")

In [8]:
print(flights[["ORIGIN_AIRPORT", "DESTINATION_AIRPORT"]].dtypes)

ORIGIN_AIRPORT         string
DESTINATION_AIRPORT    string
dtype: object


In [10]:
import pandas as pd
import numpy as np

file_path = r"C:\Users\Shakthi\Desktop\Flitzz\data\flights.csv"

flights = pd.read_csv(
    file_path,
    dtype={
        "ORIGIN_AIRPORT": "string",
        "DESTINATION_AIRPORT": "string"
    }
)

print("Data loaded successfully!")
print("Rows:", flights.shape[0])
print("Columns:", flights.shape[1])

Data loaded successfully!
Rows: 5819079
Columns: 31


## Checking Data for dtype and columns of data

In [11]:
flights.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 31 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY                  int64  
 3   DAY_OF_WEEK          int64  
 4   AIRLINE              str    
 5   FLIGHT_NUMBER        int64  
 6   TAIL_NUMBER          str    
 7   ORIGIN_AIRPORT       string 
 8   DESTINATION_AIRPORT  string 
 9   SCHEDULED_DEPARTURE  int64  
 10  DEPARTURE_TIME       float64
 11  DEPARTURE_DELAY      float64
 12  TAXI_OUT             float64
 13  WHEELS_OFF           float64
 14  SCHEDULED_TIME       float64
 15  ELAPSED_TIME         float64
 16  AIR_TIME             float64
 17  DISTANCE             int64  
 18  WHEELS_ON            float64
 19  TAXI_IN              float64
 20  SCHEDULED_ARRIVAL    int64  
 21  ARRIVAL_TIME         float64
 22  ARRIVAL_DELAY        float64
 23  DIVERTED             int64  
 24  CANCELLED

### Null Count Report for each column

In [12]:
missing = pd.DataFrame({
    "missing_count": flights.isnull().sum(),
    "missing_percentage": (flights.isnull().mean() * 100).round(2)
})

missing = missing.sort_values(
    by="missing_percentage",
    ascending=False
)

missing

,missing_count,missing_percentage
CANCELLATION_REASON,5729195,98.46
LATE_AIRCRAFT_DELAY,4755640,81.72
WEATHER_DELAY,4755640,81.72
AIRLINE_DELAY,4755640,81.72
AIR_SYSTEM_DELAY,4755640,81.72
SECURITY_DELAY,4755640,81.72
ELAPSED_TIME,105071,1.81
AIR_TIME,105071,1.81
ARRIVAL_DELAY,105071,1.81
WHEELS_ON,92513,1.59


## Removing Columns

The main reason to remove is many number of null values in the data which will lead to errors while training the model so we are removing it entirely

In [13]:
columns_to_remove = [
    "CANCELLATION_REASON",
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY"
]

flights.drop(columns=columns_to_remove, inplace=True)

print("Removed columns:")
print(columns_to_remove)

print("\nRemaining columns:", flights.shape[1])

Removed columns:
['CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']

Remaining columns: 25


Now we'll remove the rows in which we don't know the actual arrival delay which are nothing but flights which are cancled or not yet landed and also we do the same thing for scheduled flight like this means the flights which are canceled. And seperately we are removing the flights which are cancelled and diverted

In [14]:
# Remove rows where we don't know the actual arrival delay
flights.dropna(
    subset=["ARRIVAL_DELAY"],
    inplace=True
)

# Remove the 6 rows where scheduled flight time is missing
flights.dropna(
    subset=["SCHEDULED_TIME"],
    inplace=True
)

# Remove cancelled and diverted flights
flights = flights[
    (flights["CANCELLED"] == 0) &
    (flights["DIVERTED"] == 0)
].copy()

print("Rows after cleaning:", len(flights))
print("Columns:", len(flights.columns))

Rows after cleaning: 5714008
Columns: 25


## Delay Column creation

Now here we are creating a new column named Delayed_15, where the it containes 0 or 1 based on data of flights delayed or not

In [15]:
# Create the target variable
flights["DELAYED_15"] = (
    flights["ARRIVAL_DELAY"] > 15
).astype("int8")

print(flights["DELAYED_15"].value_counts())

DELAYED_15
0    4690510
1    1023498
Name: count, dtype: int64


In [16]:
flights.info()

<class 'pandas.DataFrame'>
Index: 5714008 entries, 0 to 5819078
Data columns (total 26 columns):
 #   Column               Dtype  
---  ------               -----  
 0   YEAR                 int64  
 1   MONTH                int64  
 2   DAY                  int64  
 3   DAY_OF_WEEK          int64  
 4   AIRLINE              str    
 5   FLIGHT_NUMBER        int64  
 6   TAIL_NUMBER          str    
 7   ORIGIN_AIRPORT       string 
 8   DESTINATION_AIRPORT  string 
 9   SCHEDULED_DEPARTURE  int64  
 10  DEPARTURE_TIME       float64
 11  DEPARTURE_DELAY      float64
 12  TAXI_OUT             float64
 13  WHEELS_OFF           float64
 14  SCHEDULED_TIME       float64
 15  ELAPSED_TIME         float64
 16  AIR_TIME             float64
 17  DISTANCE             int64  
 18  WHEELS_ON            float64
 19  TAXI_IN              float64
 20  SCHEDULED_ARRIVAL    int64  
 21  ARRIVAL_TIME         float64
 22  ARRIVAL_DELAY        float64
 23  DIVERTED             int64  
 24  CANCELLED     

In [17]:
flights.head()


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,DELAYED_15
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,169.0,1448,404.0,4.0,430,408.0,-22.0,0,0,0
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,263.0,2330,737.0,4.0,750,741.0,-9.0,0,0,0
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,266.0,2296,800.0,11.0,806,811.0,5.0,0,0,0
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,258.0,2342,748.0,8.0,805,756.0,-9.0,0,0,0
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,199.0,1448,254.0,5.0,320,259.0,-21.0,0,0,0


In [18]:
flights.tail()


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,DELAYED_15
5819074,2015,12,31,4,B6,688,N657JB,LAX,BOS,2359,...,272.0,2611,749.0,4.0,819,753.0,-26.0,0,0,0
5819075,2015,12,31,4,B6,745,N828JB,JFK,PSE,2359,...,195.0,1617,427.0,3.0,446,430.0,-16.0,0,0,0
5819076,2015,12,31,4,B6,1503,N913JB,JFK,SJU,2359,...,197.0,1598,424.0,8.0,440,432.0,-8.0,0,0,0
5819077,2015,12,31,4,B6,333,N527JB,MCO,SJU,2359,...,144.0,1189,327.0,3.0,340,330.0,-10.0,0,0,0
5819078,2015,12,31,4,B6,839,N534JB,JFK,BQN,2359,...,189.0,1576,437.0,5.0,440,442.0,2.0,0,0,0


## Splitting HH:MM into seperate columns

Here we have split the hours into a column and minutes into a column with normal data distribution like %100 for last 2 digits and //100 for first 2 digits

In [19]:
# Convert scheduled departure into hour and minute

flights["DEPARTURE_HOUR"] = flights["SCHEDULED_DEPARTURE"] // 100
flights["DEPARTURE_MINUTE"] = flights["SCHEDULED_DEPARTURE"] % 100

# Convert scheduled arrival into hour and minute

flights["ARRIVAL_HOUR"] = flights["SCHEDULED_ARRIVAL"] // 100
flights["ARRIVAL_MINUTE"] = flights["SCHEDULED_ARRIVAL"] % 100

In [20]:
print(
    flights[
        [
            "SCHEDULED_DEPARTURE",
            "DEPARTURE_HOUR",
            "DEPARTURE_MINUTE",
            "SCHEDULED_ARRIVAL",
            "ARRIVAL_HOUR",
            "ARRIVAL_MINUTE"
        ]
    ].head(10)
)

   SCHEDULED_DEPARTURE  DEPARTURE_HOUR  DEPARTURE_MINUTE  SCHEDULED_ARRIVAL  \
0                    5               0                 5                430   
1                   10               0                10                750   
2                   20               0                20                806   
3                   20               0                20                805   
4                   25               0                25                320   
5                   25               0                25                602   
6                   25               0                25                526   
7                   30               0                30                803   
8                   30               0                30                545   
9                   30               0                30                711   

   ARRIVAL_HOUR  ARRIVAL_MINUTE  
0             4              30  
1             7              50  
2             8             

## Creating column for period wise 

Now we are creating a new column named DEPARTURE_PERIOD where we are creating a column to understand like flights at evening may get delayed and morning flights are less delayed this feature is important for model training puposes

In [21]:
def get_departure_period(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

flights["DEPARTURE_PERIOD"] = flights["DEPARTURE_HOUR"].apply(
    get_departure_period
)

In [22]:
print(flights["DEPARTURE_PERIOD"].value_counts())

DEPARTURE_PERIOD
Morning      2341610
Afternoon    1720139
Evening      1287578
Night         364681
Name: count, dtype: int64


## New column IS_WEEKEND

Now here we created a column to represent whether the specific day is a weekend or not which can tell about the delay reasons since weekends will have croud

In [14]:
flights["IS_WEEKEND"] = (
    flights["DAY_OF_WEEK"] >= 6
).astype("int8")

## New Column SEASON

Season/Climate is one of the most important factors causing delays so here we have created season column based on months like there is a usual speculation of this month will rain and this month will be windy based on those analysis this column is created

In [15]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

flights["SEASON"] = flights["MONTH"].apply(get_season)

In [17]:
print("Final rows:", len(flights))
print("Final columns:", len(flights.columns))

print("\nColumns:")
print(flights.columns.tolist())

print("\nTarget:")
print(flights["DELAYED_15"].value_counts())

print("\nMissing values:")
print(
    flights.isnull()
    .sum()
    .sort_values(ascending=False)
)

Final rows: 5714008
Final columns: 33

Columns:
['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'DELAYED_15', 'DEPARTURE_HOUR', 'DEPARTURE_MINUTE', 'ARRIVAL_HOUR', 'ARRIVAL_MINUTE', 'IS_WEEKEND', 'SEASON', 'DEPARTURE_PERIOD']

Target:
DELAYED_15
0    4690510
1    1023498
Name: count, dtype: int64

Missing values:
YEAR                   0
MONTH                  0
DAY                    0
DAY_OF_WEEK            0
AIRLINE                0
FLIGHT_NUMBER          0
TAIL_NUMBER            0
ORIGIN_AIRPORT         0
DESTINATION_AIRPORT    0
SCHEDULED_DEPARTURE    0
DEPARTURE_TIME         0
DEPARTURE_DELAY        0
TAXI_OUT               0
WHEELS_OFF             0
SCHEDULED_TI

In [24]:
flights.head(20)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,DIVERTED,CANCELLED,DELAYED_15,DEPARTURE_HOUR,DEPARTURE_MINUTE,ARRIVAL_HOUR,ARRIVAL_MINUTE,IS_WEEKEND,SEASON,DEPARTURE_PERIOD
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,0,0,0,0,5,4,30,0,Winter,Night
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,0,0,0,0,10,7,50,0,Winter,Night
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,0,0,0,0,20,8,6,0,Winter,Night
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,0,0,0,0,20,8,5,0,Winter,Night
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,0,0,0,0,25,3,20,0,Winter,Night
5,2015,1,1,4,DL,806,N3730B,SFO,MSP,25,...,0,0,0,0,25,6,2,0,Winter,Night
6,2015,1,1,4,NK,612,N635NK,LAS,MSP,25,...,0,0,0,0,25,5,26,0,Winter,Night
7,2015,1,1,4,US,2013,N584UW,LAX,CLT,30,...,0,0,0,0,30,8,3,0,Winter,Night
8,2015,1,1,4,AA,1112,N3LAAA,SFO,DFW,30,...,0,0,0,0,30,5,45,0,Winter,Night
9,2015,1,1,4,DL,1173,N826DN,LAS,ATL,30,...,0,0,0,0,30,7,11,0,Winter,Night


## ROUTE Column Creation

This is done for mapping the origin and destination like this ANC_SEA

In [7]:
flights["ROUTE"] = (
    flights["ORIGIN_AIRPORT"].astype(str)
    + "_"
    + flights["DESTINATION_AIRPORT"].astype(str)
)

In [8]:
flights[
    [
        "ORIGIN_AIRPORT",
        "DESTINATION_AIRPORT",
        "ROUTE"
    ]
].head(10)

,ORIGIN_AIRPORT,DESTINATION_AIRPORT,ROUTE
0,ANC,SEA,ANC_SEA
1,LAX,PBI,LAX_PBI
2,SFO,CLT,SFO_CLT
3,LAX,MIA,LAX_MIA
4,SEA,ANC,SEA_ANC
5,SFO,MSP,SFO_MSP
6,LAS,MSP,LAS_MSP
7,LAX,CLT,LAX_CLT
8,SFO,DFW,SFO_DFW
9,LAS,ATL,LAS_ATL


In [9]:
print("Unique routes:", flights["ROUTE"].nunique())

Unique routes: 8570


In [10]:
print(
    flights["ROUTE"]
    .value_counts()
    .head(20)
)

ROUTE
SFO_LAX    13400
LAX_SFO    13109
JFK_LAX    11853
LAX_JFK    11851
LAS_LAX     9651
LAX_LAS     9522
LGA_ORD     9179
ORD_LGA     9101
SFO_JFK     8306
JFK_SFO     8301
OGG_HNL     8268
HNL_OGG     8246
ATL_MCO     8148
MCO_ATL     8144
LAX_ORD     8131
ATL_LGA     7955
LGA_ATL     7953
SFO_LAS     7945
ORD_LAX     7826
LAS_SFO     7812
Name: count, dtype: int64


In [11]:
flights.head()

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,CANCELLED,DELAYED_15,DEPARTURE_HOUR,DEPARTURE_MINUTE,ARRIVAL_HOUR,ARRIVAL_MINUTE,IS_WEEKEND,SEASON,DEPARTURE_PERIOD,ROUTE
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,0,0,0,5,4,30,0,Winter,Night,ANC_SEA
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,0,0,0,10,7,50,0,Winter,Night,LAX_PBI
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,0,0,0,20,8,6,0,Winter,Night,SFO_CLT
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,0,0,0,20,8,5,0,Winter,Night,LAX_MIA
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,0,0,0,25,3,20,0,Winter,Night,SEA_ANC


Saving this data as a checkpoint so we don't have to process it again and again and it gets saved as a temp copy


In [12]:
import pandas as pd
import numpy as np

checkpoint_path = r"C:\Users\Shakthi\checkpoints\flights_cleaned_time_features.csv"

flights = pd.read_csv(
    checkpoint_path,
    dtype={
        "AIRLINE": "string",
        "TAIL_NUMBER": "string",
        "ORIGIN_AIRPORT": "string",
        "DESTINATION_AIRPORT": "string",
        "SEASON": "string",
        "DEPARTURE_PERIOD": "string",
        "ROUTE": "string"
    }
)

print("Loaded successfully!")
print("Shape:", flights.shape)

Loaded successfully!
Shape: (5714008, 33)


## Class imbalance check 

This is to ensure that the output class is balanced properly

In [13]:
print(flights["DELAYED_15"].value_counts())

DELAYED_15
0    4690510
1    1023498
Name: count, dtype: int64


from the above output we can see that 0 - NO delay is very high which is imbalanced which may lead the model to  become biased towards predicting "No Delay" because that’s the majority class. But before that we have to check the percentage which is a better way than checking it as count

In [ ]:
print(
    flights["DELAYED_15"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DELAYED_15
0    82.09
1    17.91
Name: proportion, dtype: float64


Seems like 82% it will be getting highly biased so doesnt seem like this issue is fixed, but at some cases the model will work properly so running it will make the answer clear but this might work or may not since its highly imbalanced

### Sorting and Resetting Index in Pandas

Sorting them makes the time to come in chronological order to maintain the time of the data

In [15]:
flights = flights.sort_values(
    ["YEAR", "MONTH", "DAY", "SCHEDULED_DEPARTURE"]
).reset_index(drop=True)

This code is building a proper datetime column for each flight’s scheduled departure by combining YEAR, MONTH, DAY, and SCHEDULED_DEPARTURE into one timestamp. This is not necessary for model training just for understanding purposes

In [22]:
flights["FLIGHT_DATETIME"] = (
    pd.to_datetime(
        flights[["YEAR", "MONTH", "DAY"]]
    )
    + pd.to_timedelta(
        flights["SCHEDULED_DEPARTURE"] // 100,
        unit="h"
    )
    + pd.to_timedelta(
        flights["SCHEDULED_DEPARTURE"] % 100,
        unit="m"
    )
)

In [23]:
flights = flights.sort_values(
    ["FLIGHT_DATETIME", "FLIGHT_NUMBER"]
).reset_index(drop=True)

print("Flights sorted chronologically.")

Flights sorted chronologically.


In [24]:
flights.head()


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,DELAYED_15,DEPARTURE_HOUR,DEPARTURE_MINUTE,ARRIVAL_HOUR,ARRIVAL_MINUTE,IS_WEEKEND,SEASON,DEPARTURE_PERIOD,ROUTE,FLIGHT_DATETIME
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,0,0,5,4,30,0,Winter,Night,ANC_SEA,2015-01-01 00:05:00
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,0,0,10,7,50,0,Winter,Night,LAX_PBI,2015-01-01 00:10:00
2,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,0,0,20,8,5,0,Winter,Night,LAX_MIA,2015-01-01 00:20:00
3,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,0,0,20,8,6,0,Winter,Night,SFO_CLT,2015-01-01 00:20:00
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,0,0,25,3,20,0,Winter,Night,SEA_ANC,2015-01-01 00:25:00


New feature(Column) that counts how many flights have already appeared for each route up to that row. 

In [25]:
flights["ROUTE_PREVIOUS_FLIGHTS"] = (
    flights.groupby(
        "ROUTE",
        sort=False
    ).cumcount()
)

A feature that tracks how many delays have already happened on a given route before each flight. 

In [26]:
route_cumulative_delays = (
    flights.groupby(
        "ROUTE",
        sort=False
    )["DELAYED_15"]
    .cumsum()
)

flights["ROUTE_PREVIOUS_DELAYS"] = (
    route_cumulative_delays
    - flights["DELAYED_15"]
)

## Historical delay rate for each route up to the current flight

In [27]:
flights["ROUTE_DELAY_RATE"] = np.where(
    flights["ROUTE_PREVIOUS_FLIGHTS"] > 0,
    flights["ROUTE_PREVIOUS_DELAYS"]
    / flights["ROUTE_PREVIOUS_FLIGHTS"],
    np.nan
)

In [29]:
print(
    flights[
        [
            "FLIGHT_DATETIME",
            "ROUTE",
            "DELAYED_15",
            "ROUTE_PREVIOUS_FLIGHTS",
            "ROUTE_PREVIOUS_DELAYS",
            "ROUTE_DELAY_RATE"
        ]
    ].tail(20)
)

            FLIGHT_DATETIME    ROUTE  DELAYED_15  ROUTE_PREVIOUS_FLIGHTS  \
5713988 2015-12-31 23:59:00  RNO_JFK           0                     165   
5713989 2015-12-31 23:59:00  DEN_JFK           0                     806   
5713990 2015-12-31 23:59:00  LAX_ORD           0                    8130   
5713991 2015-12-31 23:59:00  DEN_TPA           0                    1575   
5713992 2015-12-31 23:59:00  MCO_SJU           0                    3536   
5713993 2015-12-31 23:59:00  SEA_IAH           0                    2175   
5713994 2015-12-31 23:59:00  DEN_RDU           0                     895   
5713995 2015-12-31 23:59:00  LAS_IAD           0                    1159   
5713996 2015-12-31 23:59:00  PDX_IAH           0                     765   
5713997 2015-12-31 23:59:00  DEN_MCO           0                    2872   
5713998 2015-12-31 23:59:00  LAX_BOS           0                    3994   
5713999 2015-12-31 23:59:00  DEN_RSW           0                     379   
5714000 2015

In [30]:
print(
    flights["ROUTE_DELAY_RATE"].describe()
)

count    5.705438e+06
mean     1.941871e-01
std      8.224350e-02
min      0.000000e+00
25%      1.438683e-01
50%      1.878652e-01
75%      2.352941e-01
max      1.000000e+00
Name: ROUTE_DELAY_RATE, dtype: float64


In [32]:
route_check = flights[
    flights["ROUTE"] == route_example
][
    [
        "FLIGHT_DATETIME",
        "ROUTE",
        "DELAYED_15",
        "ROUTE_PREVIOUS_FLIGHTS",
        "ROUTE_PREVIOUS_DELAYS",
        "ROUTE_DELAY_RATE"
    ]
].head(10)

route_check

,FLIGHT_DATETIME,ROUTE,DELAYED_15,ROUTE_PREVIOUS_FLIGHTS,ROUTE_PREVIOUS_DELAYS,ROUTE_DELAY_RATE
242,2015-01-01 06:00:00,SFO_LAX,0,0,0,NaN
658,2015-01-01 06:45:00,SFO_LAX,0,1,0,0.0
1151,2015-01-01 07:30:00,SFO_LAX,0,2,0,0.0
1530,2015-01-01 08:00:00,SFO_LAX,0,3,0,0.0
2162,2015-01-01 08:50:00,SFO_LAX,0,4,0,0.0
2270,2015-01-01 09:00:00,SFO_LAX,0,5,0,0.0
2396,2015-01-01 09:05:00,SFO_LAX,0,6,0,0.0
3040,2015-01-01 09:55:00,SFO_LAX,0,7,0,0.0
3225,2015-01-01 10:05:00,SFO_LAX,0,8,0,0.0
4036,2015-01-01 11:05:00,SFO_LAX,0,9,0,0.0


## Airline-level historical delay features 

Number of past flights, number of past delays, and delay rate, which give our model richer context about how each airline has performed historically

In [33]:
# Number of previous flights by airline
flights["AIRLINE_PREVIOUS_FLIGHTS"] = (
    flights.groupby(
        "AIRLINE",
        sort=False
    ).cumcount()
)

# Cumulative delays by airline
airline_cumulative_delays = (
    flights.groupby(
        "AIRLINE",
        sort=False
    )["DELAYED_15"]
    .cumsum()
)

# Remove current flight's outcome
flights["AIRLINE_PREVIOUS_DELAYS"] = (
    airline_cumulative_delays
    - flights["DELAYED_15"]
)

# Historical airline delay rate
flights["AIRLINE_DELAY_RATE"] = np.where(
    flights["AIRLINE_PREVIOUS_FLIGHTS"] > 0,
    flights["AIRLINE_PREVIOUS_DELAYS"]
    / flights["AIRLINE_PREVIOUS_FLIGHTS"],
    np.nan
)

## Simple Base checks like info describe head and so on

In [35]:
flights[
    [
        "FLIGHT_DATETIME",
        "AIRLINE",
        "DELAYED_15",
        "AIRLINE_PREVIOUS_FLIGHTS",
        "AIRLINE_PREVIOUS_DELAYS",
        "AIRLINE_DELAY_RATE"
    ]
].tail(20)

,FLIGHT_DATETIME,AIRLINE,DELAYED_15,AIRLINE_PREVIOUS_FLIGHTS,AIRLINE_PREVIOUS_DELAYS,AIRLINE_DELAY_RATE
5713988,2015-12-31 23:59:00,B6,0,262033,57255,0.218503
5713989,2015-12-31 23:59:00,B6,0,262034,57255,0.218502
5713990,2015-12-31 23:59:00,AA,0,712933,125238,0.175666
5713991,2015-12-31 23:59:00,F9,0,90086,22851,0.253658
5713992,2015-12-31 23:59:00,B6,0,262035,57255,0.218501
5713993,2015-12-31 23:59:00,UA,0,507758,101303,0.199510
5713994,2015-12-31 23:59:00,F9,0,90087,22851,0.253655
5713995,2015-12-31 23:59:00,UA,0,507759,101303,0.199510
5713996,2015-12-31 23:59:00,UA,0,507760,101303,0.199510
5713997,2015-12-31 23:59:00,F9,0,90088,22851,0.253652


In [36]:
airline_check = flights[
    flights["AIRLINE"] == "AA"
][
    [
        "FLIGHT_DATETIME",
        "AIRLINE",
        "DELAYED_15",
        "AIRLINE_PREVIOUS_FLIGHTS",
        "AIRLINE_PREVIOUS_DELAYS",
        "AIRLINE_DELAY_RATE"
    ]
].head(10)

airline_check

,FLIGHT_DATETIME,AIRLINE,DELAYED_15,AIRLINE_PREVIOUS_FLIGHTS,AIRLINE_PREVIOUS_DELAYS,AIRLINE_DELAY_RATE
1,2015-01-01 00:10:00,AA,0,0,0,NaN
2,2015-01-01 00:20:00,AA,0,1,0,0.0
7,2015-01-01 00:30:00,AA,0,2,0,0.0
12,2015-01-01 00:35:00,AA,0,3,0,0.0
21,2015-01-01 01:00:00,AA,0,4,0,0.0
23,2015-01-01 01:05:00,AA,0,5,0,0.0
29,2015-01-01 01:20:00,AA,0,6,0,0.0
31,2015-01-01 01:27:00,AA,0,7,0,0.0
62,2015-01-01 05:10:00,AA,0,8,0,0.0
67,2015-01-01 05:15:00,AA,1,9,0,0.0


In [37]:
flights.info()


<class 'pandas.DataFrame'>
RangeIndex: 5714008 entries, 0 to 5714007
Data columns (total 41 columns):
 #   Column                    Dtype         
---  ------                    -----         
 0   YEAR                      int64         
 1   MONTH                     int64         
 2   DAY                       int64         
 3   DAY_OF_WEEK               int64         
 4   AIRLINE                   string        
 5   FLIGHT_NUMBER             int64         
 6   TAIL_NUMBER               string        
 7   ORIGIN_AIRPORT            string        
 8   DESTINATION_AIRPORT       string        
 9   SCHEDULED_DEPARTURE       int64         
 10  DEPARTURE_TIME            float64       
 11  DEPARTURE_DELAY           float64       
 12  TAXI_OUT                  float64       
 13  WHEELS_OFF                float64       
 14  SCHEDULED_TIME            float64       
 15  ELAPSED_TIME              float64       
 16  AIR_TIME                  float64       
 17  DISTANCE           

In [38]:
import os

output_path = r"C:\Users\Shakthi\Desktop\Flitzz\data\flights_feature_engineered.csv"

flights.to_csv(
    output_path,
    index=False
)

print("Updated flights file saved successfully!")
print("Path:", output_path)

Updated flights file saved successfully!
Path: C:\Users\Shakthi\Desktop\Flitzz\data\flights_feature_engineered.csv
